In [8]:
##DEPENDENCIAS
!pip install transformers torch pymongo python-dotenv scikit-learn plotly

In [9]:
##IMPORTACIONES Y CONEION A MONGO
from transformers import BertTokenizer, BertModel, pipeline
import torch
import pandas as pd
import numpy as np
from pymongo import MongoClient
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
import plotly.express as px
import os
import re

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")
client = MongoClient(MONGO_URI)
db = client["proyecto2_musica"]
coleccion = db["canciones"]

print("Conectado a MongoDB Atlas")
print(f"Total canciones: {coleccion.count_documents({})}")

Conectado a MongoDB Atlas
Total canciones: 10420


In [10]:
##CARGAR BETO
print("Cargando BETO desde HuggingFace...")

modelo_beto = "dccuchile/bert-base-spanish-wwm-cased"

tokenizer = BertTokenizer.from_pretrained(modelo_beto)
model = BertModel.from_pretrained(modelo_beto)
model.eval()

print("BETO cargado correctamente")
print(f"Parametros del modelo: {sum(p.numel() for p in model.parameters()):,}")

Cargando BETO desde HuggingFace...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 35772.70it/s]
BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly 

BETO cargado correctamente
Parametros del modelo: 109,850,880


In [11]:
##CARGAR CORPUS DE MONGO
print("Cargando canciones en español desde MongoDB...")

canciones = list(coleccion.find(
    {
        "letra": {"$exists": True, "$ne": None},
        "idioma": "es"
    },
    {"letra": 1, "genero": 1, "artista": 1, "titulo": 1, "_id": 0}
))

df = pd.DataFrame(canciones)
print(f"{len(df)} canciones en español cargadas")
print(df["genero"].value_counts())

Cargando canciones en español desde MongoDB...
9872 canciones en español cargadas
genero
pop        2457
country    1907
blues      1588
rock       1386
jazz       1307
reggae      884
hip hop     319
Name: count, dtype: int64


In [12]:
##FUNCION PARA GENERAR MEBENDINGS
def get_embedding_beto(texto, max_length=512):
    if not isinstance(texto, str) or len(texto.strip()) == 0:
        return None

    # Truncar texto si es muy largo
    texto = texto[:1000]

    inputs = tokenizer(
        texto,
        return_tensors="pt",
        max_length=max_length,
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # Vector CLS (representacion de toda la oracion)
    embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
    return embedding

# Prueba rapida
prueba = get_embedding_beto("Hoy me siento feliz cantando bajo la lluvia")
print(f"Embedding generado: shape {prueba.shape}")
print(f"Primeros 5 valores: {prueba[:5]}")

Embedding generado: shape (768,)
Primeros 5 valores: [-0.40684998 -0.914615   -0.23492159 -0.605608    0.34833848]


In [13]:
##ANALISIS DE POLISEMIA CONTEXTUAL
palabras_polisemicas = {
    "corazon": [
        "Mi corazon late muy rapido cuando te veo",
        "El medico reviso mi corazon en el hospital",
        "Corazon de piedra tiene ese hombre sin amor",
        "Escucha el corazon de la musica reggae",
        "Le di todo mi corazon y me lo rompio"
    ],
    "fuego": [
        "El fuego del amor me consume por dentro",
        "Hay fuego en el escenario esta noche de rock",
        "El fuego destruyo la casa del vecino",
        "Tienes fuego en los ojos cuando bailas",
        "El fuego sagrado del reggae nos une"
    ],
    "noche": [
        "Esta noche de amor no la olvidare jamas",
        "La noche oscura del alma me consume",
        "Noche de fiesta en el club de jazz",
        "Cada noche te extraño mas que el dia anterior",
        "La noche estrellada me recuerda tu mirada"
    ]
}

print("Analisis de polisemia contextual con BETO:\n")

for palabra, frases in palabras_polisemicas.items():
    print(f"Palabra: '{palabra}'")
    embeddings = [get_embedding_beto(frase) for frase in frases]

    similitudes = cosine_similarity(embeddings)

    for i in range(len(frases)):
        for j in range(i+1, len(frases)):
            sim = similitudes[i][j]
            print(f"  Frase {i+1} vs Frase {j+1}: {sim:.4f}")
    print()

Analisis de polisemia contextual con BETO:

Palabra: 'corazon'
  Frase 1 vs Frase 2: 0.8763
  Frase 1 vs Frase 3: 0.8694
  Frase 1 vs Frase 4: 0.8946
  Frase 1 vs Frase 5: 0.8964
  Frase 2 vs Frase 3: 0.8348
  Frase 2 vs Frase 4: 0.8458
  Frase 2 vs Frase 5: 0.8595
  Frase 3 vs Frase 4: 0.8752
  Frase 3 vs Frase 5: 0.8982
  Frase 4 vs Frase 5: 0.8656

Palabra: 'fuego'
  Frase 1 vs Frase 2: 0.8491
  Frase 1 vs Frase 3: 0.8176
  Frase 1 vs Frase 4: 0.8572
  Frase 1 vs Frase 5: 0.8524
  Frase 2 vs Frase 3: 0.8456
  Frase 2 vs Frase 4: 0.9089
  Frase 2 vs Frase 5: 0.8147
  Frase 3 vs Frase 4: 0.7956
  Frase 3 vs Frase 5: 0.7552
  Frase 4 vs Frase 5: 0.8037

Palabra: 'noche'
  Frase 1 vs Frase 2: 0.8259
  Frase 1 vs Frase 3: 0.8086
  Frase 1 vs Frase 4: 0.8666
  Frase 1 vs Frase 5: 0.8659
  Frase 2 vs Frase 3: 0.7304
  Frase 2 vs Frase 4: 0.7612
  Frase 2 vs Frase 5: 0.8623
  Frase 3 vs Frase 4: 0.8426
  Frase 3 vs Frase 5: 0.7818
  Frase 4 vs Frase 5: 0.8231



In [14]:
##BUSQUEDA SEMANTICA POR CANCIONES
print("Generando embeddings para busqueda semantica...")

# Tomar muestra de 500 canciones en español
muestra = df.sample(min(500, len(df)), random_state=42)

embeddings_canciones = []
indices_validos = []

for idx, row in muestra.iterrows():
    emb = get_embedding_beto(str(row["letra"])[:500])
    if emb is not None:
        embeddings_canciones.append(emb)
        indices_validos.append(idx)

embeddings_matrix = np.array(embeddings_canciones)
df_muestra = muestra.loc[indices_validos].reset_index(drop=True)

print(f"Embeddings generados: {len(embeddings_canciones)} canciones")

def buscar_canciones_similares(consulta, top_n=5):
    emb_consulta = get_embedding_beto(consulta)
    if emb_consulta is None:
        return []

    similitudes = cosine_similarity([emb_consulta], embeddings_matrix)[0]
    indices_top = similitudes.argsort()[-top_n:][::-1]

    print(f"Consulta: '{consulta}'")
    print(f"Canciones mas similares:")
    for i in indices_top:
        print(f"  {df_muestra.iloc[i]['titulo']} - {df_muestra.iloc[i]['artista']} (similitud: {similitudes[i]:.4f})")
    print()

buscar_canciones_similares("cancion de amor y desamor en la noche")
buscar_canciones_similares("fiesta y baile con amigos")
buscar_canciones_similares("tristeza y soledad")


Generando embeddings para busqueda semantica...
Embeddings generados: 500 canciones
Consulta: 'cancion de amor y desamor en la noche'
Canciones mas similares:
  mr collins, mr collins - albert collins (similitud: 0.8231)
  violent love - ted nugent (similitud: 0.8107)
  give it to me - homeshake (similitud: 0.7867)
  shangri-la - the lettermen (similitud: 0.7819)
  room to breathe - chase bryant (similitud: 0.7799)

Consulta: 'fiesta y baile con amigos'
Canciones mas similares:
  mr collins, mr collins - albert collins (similitud: 0.8103)
  violent love - ted nugent (similitud: 0.7941)
  my bucket's got a hole in it - louis armstrong (similitud: 0.7800)
  seven below - phish (similitud: 0.7777)
  love me, love me - ben e. king (similitud: 0.7712)

Consulta: 'tristeza y soledad'
Canciones mas similares:
  violent love - ted nugent (similitud: 0.7770)
  my bucket's got a hole in it - louis armstrong (similitud: 0.7631)
  mr collins, mr collins - albert collins (similitud: 0.7542)
  anoth

In [15]:
##MASKED LENGUAJE MODEL
print("Cargando pipeline de Masked Language Model...")

mlm_pipeline = pipeline(
    "fill-mask",
    model=modelo_beto,
    tokenizer=modelo_beto
)

frases_por_genero = {
    "pop": [
        f"Te quiero con todo mi {mlm_pipeline.tokenizer.mask_token}",
        f"Bailamos toda la {mlm_pipeline.tokenizer.mask_token} juntos",
    ],
    "rock": [
        f"La {mlm_pipeline.tokenizer.mask_token} nos hace libres",
        f"Gritamos con toda nuestra {mlm_pipeline.tokenizer.mask_token}",
    ],
    "reggae": [
        f"La {mlm_pipeline.tokenizer.mask_token} nos une a todos",
        f"Siente el {mlm_pipeline.tokenizer.mask_token} del reggae",
    ]
}

print("Predicciones Masked Language Model por genero:\n")

for genero, frases in frases_por_genero.items():
    print(f"Genero: {genero.upper()}")
    for frase in frases:
        resultados = mlm_pipeline(frase, top_k=3)
        predicciones = [r["token_str"] for r in resultados]
        print(f"  '{frase}' -> {predicciones}")
    print()

Cargando pipeline de Masked Language Model...


Loading weights: 100%|██████████| 204/204 [00:00<00:00, 36090.69it/s]
The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Predicciones Masked Language Model por genero:

Genero: POP
  'Te quiero con todo mi [MASK]' -> ['corazón', 'corazon', 'alma']
  'Bailamos toda la [MASK] juntos' -> ['noche', 'vida', 'tarde']

Genero: ROCK
  'La [MASK] nos hace libres' -> ['libertad', 'vida', 'que']
  'Gritamos con toda nuestra [MASK]' -> ['fuerza', 'alma', 'energía']

Genero: REGGAE
  'La [MASK] nos une a todos' -> ['vida', 'amistad', 'música']
  'Siente el [MASK] del reggae' -> ['sonido', 'ritmo', 'espíritu']



In [16]:
##VISUALIZACIONES TSNE Y EMBENDINGS BETO
print("Generando visualizacion t-SNE de embeddings BETO...")

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(embeddings_matrix)

df_muestra["x"] = embeddings_2d[:, 0]
df_muestra["y"] = embeddings_2d[:, 1]
df_muestra["genero"] = df_muestra["genero"].fillna("scraping")

fig = px.scatter(
    df_muestra,
    x="x",
    y="y",
    color="genero",
    hover_data=["titulo", "artista"],
    title="t-SNE - Embeddings BETO por genero",
    width=900,
    height=600
)
fig.show()

Generando visualizacion t-SNE de embeddings BETO...


In [17]:
##CERRAR CONEXCIONES
client.close()
print("Conexion cerrada")

Conexion cerrada
